# Bigram Model — Gradient-Descent Optimization

A companion to [`bigrams.ipynb`](bigrams.ipynb). There, a single linear layer is
trained with plain SGD at a fixed `lr = 50` to recover the frequentist bigram
model. That one rate is enough to tell the main story, but it hides a lot: *why*
50, what happens at other rates, whether a schedule or a modern optimizer does
better, and why the gradient never quite settles to zero.

This notebook explores those questions on **exactly the same model and data**:

1. **Effect of the learning rate** — sweep a range of fixed rates.
2. **Learning-rate schedules** — can decaying the rate escape the plateau?
3. **Popular optimizers** — how do SGD+Nesterov, RMSprop, Adam and NAdam compare?

The narrative writeup lives in
[The Stubborn Gradient](../docs/stubborn_gradient.html).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

from bznames.ibge import load_ibge_name_data
from bznames.linear_model import compute_linear_nn_nll_for_tokens
from bznames.metrics import compute_bigram_nll_for_tokens
from bznames.tokenizer import CharacterEncoder, tokenize_dataset

%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'

## Setup

Rebuild the pieces the experiments depend on: the bigram-tokenized dataset, the
frequentist bigram model (the loss **floor** the NN is chasing) and the uniform
model (the **ceiling**). The per-example NLL used for training is imported from
`bznames.linear_model` so it stays identical to the main notebook.

In [ ]:
data = load_ibge_name_data()

SPECIAL_TOKEN = "."
names = [x["name"] for x in data]
encoder = CharacterEncoder.from_words(names, special_token=SPECIAL_TOKEN)
vocab_size = encoder.vocab_size

# Bigram (order-2) tokenization
input_tokens, output_tokens, freqs = tokenize_dataset(
    data, encoder, ngram_size=2, show_progress=True
)
input_tokens = torch.tensor(input_tokens, dtype=torch.long)
output_tokens = torch.tensor(output_tokens, dtype=torch.long)
freqs = torch.tensor(freqs, dtype=torch.long)
sample_weights = freqs / freqs.sum()

# Frequentist bigram model (Laplace-smoothed) and the uniform baseline
counts = torch.zeros((vocab_size, vocab_size), dtype=torch.long)
counts.index_put_((input_tokens[:, 0], output_tokens), freqs, accumulate=True)
counts += 1
bigram_cond_probs = counts / counts.sum(dim=1, keepdim=True)

uniform_probs = torch.ones_like(counts, dtype=torch.float32)
uniform_probs /= uniform_probs.sum(dim=1, keepdim=True)

# Reference NLLs: the bigram floor and the uniform ceiling
bigram_nll = compute_bigram_nll_for_tokens(
    bigram_cond_probs, input_tokens, output_tokens, sample_weights
)
uniform_nll = compute_bigram_nll_for_tokens(
    uniform_probs, input_tokens, output_tokens, sample_weights
)

print(f"{vocab_size=}  bigram_nll={bigram_nll:.4f}  uniform_nll={uniform_nll:.4f}")

## 1. Effect of the learning rate

Sweep a range of fixed learning rates from the same initial weights. The loss
trajectory shows how fast each rate descends; the gradient norm (log scale)
shows whether it actually comes to rest or oscillates on a plateau.

In [ ]:
# Compare training trajectories across different learning rates
def train_and_track_loss(
    learning_rate: float, epochs: int = 100, progress: bool = True
) -> tuple[list[float], list[float]]:
    # Re-seed so every run starts from the same initial weights (fair comparison)
    gen = torch.Generator().manual_seed(42)
    W = torch.randn((vocab_size, vocab_size), generator=gen, requires_grad=True)

    loss_values, grad_norms = [], []
    # Inner bar clears itself (leave=False) once the run for this lr completes
    epoch_bar = tqdm(
        range(epochs + 1),
        desc=f"lr={learning_rate}",
        leave=False,
        disable=not progress,
    )
    for _ in epoch_bar:
        loss = compute_linear_nn_nll_for_tokens(W, input_tokens, output_tokens, sample_weights)
        loss_values.append(loss.item())

        W.grad = None
        loss.backward()
        # ||grad||_2 for the same W state as this epoch's loss
        grad_norms.append(W.grad.norm().item())
        epoch_bar.set_postfix(loss=f"{loss.item():.3f}")

        with torch.no_grad():
            W -= learning_rate * W.grad

    return loss_values, grad_norms


learning_rates = [1, 10, 25, 50, 75, 100]
trajectories = {lr: train_and_track_loss(lr) for lr in tqdm(learning_rates, desc="learning rates")}

fig, (ax_loss, ax_grad) = plt.subplots(1, 2, figsize=(13, 5), sharex=True, constrained_layout=True)

# Encode the (ordered) learning rate with a perceptual colormap: dark -> bright = low -> high.
# Cap below 1.0 so the brightest line avoids the low-contrast pure yellow on white.
colors = plt.cm.viridis(np.linspace(0.0, 0.85, len(learning_rates)))
for (lr, (loss_values, grad_norms)), color in zip(trajectories.items(), colors, strict=True):
    ax_loss.plot(loss_values, color=color, linewidth=1.5)
    ax_grad.plot(grad_norms, color=color, linewidth=1.5, label=f"{lr}")

# Left: loss trajectory with neutral reference baselines
ax_loss.axhline(bigram_nll, color="black", linestyle="--", linewidth=1, label="bigram model")
ax_loss.axhline(uniform_nll, color="gray", linestyle=":", linewidth=1, label="uniform model")
ax_loss.set_ylabel("loss (NLL)")
ax_loss.set_title("Loss trajectory")
ax_loss.legend(loc="upper right", frameon=False)

# Right: gradient norm (log scale, since it spans orders of magnitude across lr)
ax_grad.set_yscale("log")
ax_grad.set_ylabel(r"gradient L2 norm  $\|\nabla W\|_2$")
ax_grad.set_title("Gradient magnitude")
ax_grad.legend(title="learning rate", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

for ax in (ax_loss, ax_grad):
    ax.set_xlim(0, len(next(iter(trajectories.values()))[0]) - 1)
    ax.set_xlabel("epoch")
    ax.grid(True, alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Effect of learning rate on training dynamics");

## 2. Learning-rate schedules

`lr = 50` reaches the valley floor fast but then leaves a stubborn gradient
plateau. Can a schedule — start large, then decay into the contraction regime —
damp that oscillation and drive the gradient toward zero, something no single
fixed rate can do at once?

In [ ]:
# Can a learning-rate schedule escape the "stubborn gradient" plateau?
# Idea: start with a large lr to reach the valley floor fast, then decay it into
# the contraction regime to damp the oscillation and drive the
# gradient toward zero -- something no single fixed lr can do at once.
from collections.abc import Callable


def train_with_schedule(
    schedule: Callable[[int, int], float], epochs: int = 200, progress: bool = True
) -> tuple[list[float], list[float], list[float]]:
    # Re-seed so every run starts from the same initial weights (fair comparison)
    gen = torch.Generator().manual_seed(42)
    W = torch.randn((vocab_size, vocab_size), generator=gen, requires_grad=True)

    loss_values, grad_norms, lr_values = [], [], []
    # Inner bar clears itself (leave=False) once this schedule's run completes
    epoch_bar = tqdm(range(epochs + 1), leave=False, disable=not progress)
    for t in epoch_bar:
        lr = schedule(t, epochs)
        lr_values.append(lr)

        loss = compute_linear_nn_nll_for_tokens(W, input_tokens, output_tokens, sample_weights)
        loss_values.append(loss.item())

        W.grad = None
        loss.backward()
        # ||grad||_2 for the same W state as this epoch's loss
        grad_norms.append(W.grad.norm().item())
        epoch_bar.set_postfix(lr=f"{lr:.2f}", loss=f"{loss.item():.3f}")

        with torch.no_grad():
            W -= lr * W.grad

    return loss_values, grad_norms, lr_values


# Schedules map (epoch, total_epochs) -> learning rate. All peak at the fixed lr
# that reached the lowest loss yet left a stubborn gradient plateau (lr=50).
def constant(lr0: float) -> Callable[[int, int], float]:
    return lambda t, total: lr0


def step_decay(
    lr0: float, milestones: tuple[float, ...], gamma: float = 0.2
) -> Callable[[int, int], float]:
    # Drop lr by a factor of gamma at each fraction-of-training milestone
    return lambda t, total: lr0 * gamma ** sum(t >= f * total for f in milestones)


def exponential_decay(lr0: float, lr_final: float) -> Callable[[int, int], float]:
    # Geometric interpolation lr0 -> lr_final across the full run
    return lambda t, total: lr0 * (lr_final / lr0) ** (t / total)


def cosine_decay(lr0: float, lr_final: float = 0.0) -> Callable[[int, int], float]:
    def sched(t: int, total: int) -> float:
        progress = 0.5 * (1 + np.cos(np.pi * t / total))
        return float(lr_final + (lr0 - lr_final) * progress)

    return sched


peak_lr = 50
# Long horizon: the decayed schedules need room to grind below the fixed-lr floor
n_epochs = 1000
schedules = {
    "constant 50": constant(peak_lr),
    # Step drops kick in late (epochs 750, 875), long after the loss saturates
    # (~epoch 300): hold lr=50 to ride the descent, then damp the stiff oscillation.
    "step 50->10->2": step_decay(peak_lr, milestones=(3 / 4, 7 / 8), gamma=0.2),
    "exponential 50->2": exponential_decay(peak_lr, lr_final=2),
    "cosine 50->0": cosine_decay(peak_lr, lr_final=0.0),
}
runs = {
    name: train_with_schedule(sched, epochs=n_epochs)
    for name, sched in tqdm(schedules.items(), desc="schedules")
}

# The loss ends up ~tied across schedules; the final gradient norm is what pulls
# them apart -- it measures how close each run is to actually coming to rest.
print(f"Final state after {n_epochs} epochs (bigram NLL = {bigram_nll:.4f}):")
for name, (loss_values, grad_norms, _) in runs.items():
    final_loss = loss_values[-1]
    gap = final_loss - bigram_nll
    print(
        f"  {name:<20}: loss={final_loss:.4f} (gap {gap:+.4f})  final ||grad||={grad_norms[-1]:.2e}"
    )

fig, (ax_lr, ax_loss, ax_grad) = plt.subplots(
    1, 3, figsize=(17, 4.8), sharex=True, constrained_layout=True
)

# One perceptual color per schedule (ordered: no decay -> most aggressive decay)
colors = plt.cm.viridis(np.linspace(0.0, 0.85, len(schedules)))
for (name, run), color in zip(runs.items(), colors, strict=True):
    loss_values, grad_norms, lr_values = run
    ax_lr.plot(lr_values, color=color, linewidth=1.5)
    # Plot the suboptimality gap so the log axis (below) can resolve the tail
    ax_loss.plot([v - bigram_nll for v in loss_values], color=color, linewidth=1.5)
    ax_grad.plot(grad_norms, color=color, linewidth=1.5, label=name)

# Left: the schedule itself -- what each strategy does to lr over time
ax_lr.set_ylabel("learning rate")
ax_lr.set_title("Learning-rate schedule")

# Middle: suboptimality gap on a log axis. Raw NLL squishes every curve onto the
# floor; the gap (loss - bigram_nll) on a log scale resolves the tail separation.
# The bigram model is now the zero asymptote; uniform sits far above as a ref.
ax_loss.set_yscale("log")
uniform_gap = uniform_nll - bigram_nll
ax_loss.axhline(uniform_gap, color="gray", linestyle=":", linewidth=1, label="uniform model")
ax_loss.set_ylabel(r"loss $-$ bigram NLL")
ax_loss.set_title("Loss above bigram floor")
ax_loss.legend(loc="upper right", frameon=False)

# Right: gradient norm (log scale) -- where the constant-lr plateau breaks
ax_grad.set_yscale("log")
ax_grad.set_ylabel(r"gradient L2 norm  $\|\nabla W\|_2$")
ax_grad.set_title("Gradient magnitude")
ax_grad.legend(title="schedule", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

for ax in (ax_lr, ax_loss, ax_grad):
    ax.set_xlim(0, n_epochs)
    ax.set_xlabel("epoch")
    ax.grid(True, alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Learning-rate scheduling vs. a fixed rate: escaping the stubborn gradient");

## 3. Popular optimizers

The hand-tuned SGD schedules above needed manual design. Do off-the-shelf
production optimizers escape the stubborn gradient on their own — with no lr
schedule — just by adapting per-parameter step sizes? All are seeded to the same
initial `W` and compared against the fixed rate and the step schedule from above
(reused via `runs`, since they are expensive to recompute).

In [ ]:
# How do popular production optimizers compare to the hand-tuned SGD schedules?
# All use official torch.optim implementations, seeded to the same initial W.
# Question: do they escape the stubborn gradient on their own (no lr schedule)?
def train_with_optimizer(
    make_optimizer, epochs: int = 200, progress: bool = True, label: str = ""
) -> tuple[list[float], list[float]]:
    # Same initial weights as the SGD experiments above (fair comparison)
    gen = torch.Generator().manual_seed(42)
    W = torch.randn((vocab_size, vocab_size), generator=gen, requires_grad=True)
    optimizer = make_optimizer(W)

    loss_values, grad_norms = [], []
    epoch_bar = tqdm(range(epochs + 1), desc=label, leave=False, disable=not progress)
    for _ in epoch_bar:
        loss = compute_linear_nn_nll_for_tokens(W, input_tokens, output_tokens, sample_weights)
        loss_values.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        # ||grad||_2 for the same W state as this epoch's loss
        grad_norms.append(W.grad.norm().item())
        epoch_bar.set_postfix(loss=f"{loss.item():.3f}")

        optimizer.step()

    return loss_values, grad_norms


# Each factory builds a fresh optimizer bound to W. Learning rates differ by family:
# momentum-SGD wants a large step; the adaptive methods normalize, so they want small.
optimizers = {
    "SGD+Nesterov, lr=5": lambda W: torch.optim.SGD([W], lr=5, momentum=0.9, nesterov=True),
    "RMSprop, lr=0.02": lambda W: torch.optim.RMSprop([W], lr=0.02),
    "Adam, lr=0.1": lambda W: torch.optim.Adam([W], lr=0.1),
    "NAdam, lr=0.1": lambda W: torch.optim.NAdam([W], lr=0.1),
}
opt_runs = {
    name: train_with_optimizer(make, epochs=n_epochs, label=name)
    for name, make in tqdm(optimizers.items(), desc="optimizers")
}

# Reuse the SGD runs from the previous cell (expensive to recompute) as references
constant_run = runs["constant 50"]
step_run = runs["step 50->10->2"]

print(f"Final state after {n_epochs} epochs (bigram NLL = {bigram_nll:.4f}):")
summary = {"SGD fixed lr=50": constant_run, "SGD step schedule": step_run, **opt_runs}
for name, (loss_values, grad_norms, *_) in summary.items():
    print(
        f"  {name:<20}: loss={loss_values[-1]:.4f} "
        f"(gap {loss_values[-1] - bigram_nll:+.4f})  final ||grad||={grad_norms[-1]:.2e}"
    )

fig, (ax_loss, ax_grad) = plt.subplots(1, 2, figsize=(13, 5), sharex=True, constrained_layout=True)

# References: the stubborn fixed rate (black dashed) and our best hand-tuned schedule (gray)
c_loss, c_grad, *_ = constant_run
s_loss, s_grad, *_ = step_run
ax_loss.plot([v - bigram_nll for v in c_loss], color="black", linestyle="--", linewidth=1)
ax_grad.plot(c_grad, color="black", linestyle="--", linewidth=1, label="fixed lr=50")
ax_loss.plot([v - bigram_nll for v in s_loss], color="0.6", linewidth=1)
ax_grad.plot(s_grad, color="0.6", linewidth=1, label="step schedule")

# Popular DL optimizers in a perceptual colormap
colors = plt.cm.viridis(np.linspace(0.0, 0.85, len(opt_runs)))
for (name, (loss_values, grad_norms)), color in zip(opt_runs.items(), colors, strict=True):
    ax_loss.plot([v - bigram_nll for v in loss_values], color=color, linewidth=1.8)
    ax_grad.plot(grad_norms, color=color, linewidth=1.8, label=name)

# Left: suboptimality gap (log); bigram floor is the zero asymptote
ax_loss.set_yscale("log")
ax_loss.set_ylabel(r"loss $-$ bigram NLL")
ax_loss.set_title("Loss above bigram floor")

# Right: gradient norm (log) -- do the optimizers come to rest without a schedule?
ax_grad.set_yscale("log")
ax_grad.set_ylabel(r"gradient L2 norm  $\|\nabla W\|_2$")
ax_grad.set_title("Gradient magnitude")
ax_grad.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

for ax in (ax_loss, ax_grad):
    ax.set_xlim(0, n_epochs)
    ax.set_xlabel("epoch")
    ax.grid(True, alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Popular optimizers vs. fixed-rate and hand-tuned SGD");